# Training a model on the complete data set

This script walks through the full workflow of training a model

In [ ]:
import torch
from ddpm import NoiseScheduler, UNet, train, find_lr, generate_image, noisy_image
from ddpm.dataset import load_mnist, get_noisy_loaders
from ddpm.utils import load_unet, channel_list, model_name, path_name
from ddpm.viz import plot_generated

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 1. Noise scheduler and data

if you want to train on a subset of the data, use get_noisy_loaders_filtered

In [ ]:
scheduler = NoiseScheduler(T=1000, beta_start=1e-4, beta_end=0.02)

train_set, test_set = load_mnist()
train_loader, test_loader = get_noisy_loaders(train_set, test_set, scheduler, batch_size=32)

## 2. Build a UNet

`channels` sets the feature map depth at each encoder level.
The decoder mirrors this automatically.
`convs_per_level` is how many conv layers per resolution block.

In [ ]:
channel0 = 128
channels = channel_list(channel0)  
cpl = 2                       # convs per level

model_path = '/home/scur0036/diffusion-models-project/models/base_C0_128_convs_2wd_1e-6_long.pkl'

unet = load_unet(model_path, channels=channels, convs_per_level=cpl)

print(f"Model: {model_name(channel0, cpl)}")
print(f"Parameters: {sum(p.numel() for p in unet.parameters()):,}")

## 3. Find a learning rate

In [ ]:
suggested_lr = find_lr(unet, train_loader)
lr = suggested_lr * 0.5
print(f"Suggested LR: {suggested_lr:.2e} → using {lr:.2e}")

## 4. Train

In [ ]:
save_path = path_name(channel0, cpl,add_desc="wd_1e-6_long")   # e.g. "base_C0_64_convs_2.pkl"

train_losses, test_losses = train(
    unet, train_loader, test_loader,
    epochs=200,
    lr=lr,
    weight_decay=1e-6,
    early_stopping_patience=10,
    save_path=save_path,
)

Plot the loss curves manually (training doesn't do this anymore):

In [ ]:
import matplotlib.pyplot as plt

def plot_loss(train_losses,test_losses):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label='Train')
    plt.plot(test_losses,  label='Test')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Loss curve')
    plt.legend()
    plt.tight_layout()
    plt.show()

plot_loss(train_losses,test_losses)

## 5. Save and load

In [ ]:
unet_loaded = load_unet(save_path, channels=channels, convs_per_level=cpl)
# unet_loaded is already in eval mode and on `device`

## 6. Generate images

In [ ]:
x = generate_image(unet_loaded, scheduler, stochasticity=1.0, n_images=8)
plot_generated(x, ncol=4)

### Intermediates

In [ ]:
x_final, intermediates = generate_image(
    unet_loaded, scheduler, stochasticity=1.0, n_images=1, return_intermediates=True
)

# Plot every 100th step
fig, axes = plt.subplots(1, 11, figsize=(22, 2))
steps_to_show = list(range(0, 1000, 100)) + [999]
for ax, idx in zip(axes, steps_to_show):
    img = intermediates[idx].squeeze()
    if img.min() < 0:
        img = (img + 1) / 2
    ax.imshow(img.clamp(0, 1), cmap='gray')
    ax.set_title(f't={1000 - idx}', fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.show()